# A small training loop on geometric features

The earlier notebooks are geometry written in PyTorch. This one is the other half of a coding round that says "possibly PyTorch": a loss that backpropagates, a model that does not pretend `eigh` is part of the graph, and a training step you can talk through.

The three classes are synthetic and obvious on purpose. Ground is a flat cloud in XY. A wall is a flat cloud in YZ. A pole is a vertical line. Each cloud is centered, so the model cannot cheat by looking at where the object sits.

Float64 stays on. A production net is often float32; say that, then keep this exercise in float64 so the features and the weights match.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
torch.set_printoptions(precision=4, sci_mode=False)


## 1. Differentiable point-to-plane loss

Write `point_plane_loss(points, centroid, normal)`.

Return the mean squared signed distance from each row of `points` to the plane. Positive distance is along `normal`. The normal is not necessarily a unit vector; normalize it. A zero-length normal raises `ValueError`.

The scalar has to be differentiable in `points`. The check calls `.backward()` and expects a non-zero gradient.

This is the residual a curb or a ground fit actually cares about. Cross-entropy on a point label does not say how many centimeters the surface is off.

<details><summary>Hint</summary>

`((points - centroid) @ unit_normal).square().mean()`. Build it out of ops that autograd understands. Do not detach `points`.

</details>


In [ ]:
import torch

def point_plane_loss(points, centroid, normal):
    """Mean squared signed distance. Differentiable in `points`."""
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def signed_plane_distances(points, centroid, normal):
    """Signed distance from each point to the plane. Positive along `normal`."""
    points = _t(points)
    normal = _t(normal)
    n = torch.linalg.norm(normal)
    if n < 1e-12:
        raise ValueError("normal has zero length")
    unit = normal / n
    return (points - _t(centroid)) @ unit

def point_plane_loss(points, centroid, normal):
    """Mean squared signed distance. Differentiable in `points`."""
    return signed_plane_distances(points, centroid, normal).square().mean()

def check_point_plane_loss():
    pts = torch.tensor([[0.0, 0.0, 1.0], [0.0, 0.0, -1.0], [1.0, 0.0, 0.0]])
    loss = point_plane_loss(pts, torch.zeros(3), torch.tensor([0.0, 0.0, 2.0]))
    assert torch.allclose(loss, torch.tensor(2.0 / 3.0))
    leaf = torch.tensor([[0.0, 0.0, 1.0], [1.0, 0.0, 0.0]], requires_grad=True)
    point_plane_loss(leaf, torch.zeros(3), torch.tensor([0.0, 0.0, 1.0])).backward()
    assert leaf.grad is not None and torch.isfinite(leaf.grad).all()
    assert float(leaf.grad[0, 2]) != 0.0
    print("passed")

check_point_plane_loss()


## 2. A linear head on handcrafted features

`shape_features` is the neighborhood descriptor from the terrain notebook: linearity, planarity, sphericity, principal direction, normal. Eigenvectors come from `eigh`. Do not put that decomposition in the autograd graph. Compute the four numbers under `torch.no_grad`, then train a linear classifier on them.

Write two functions:

- `geometric_feature_row(points)` returns a length-4 tensor: linearity, planarity, `|direction · up|`, `|normal · up|`. Up is +Z.
- `train_feature_head(steps=250, seed=0)` builds a dataset, trains `nn.Linear(4, 3)`, and returns the head and the held-out accuracy.

Use the generator below. Class 0 is ground, 1 is a vertical wall, 2 is a vertical pole. Draw 48 clouds of each class, 32 points each. Hold out 25% with `torch.randperm` from the same generator. Adam, learning rate `0.08`, cross-entropy, `steps` updates on the full training set (no mini-batches). The check requires held-out accuracy above 0.9.

Say why this split is legal here and illegal on a real job. These clouds are independent draws. A real cloud has to be split by project or by neighborhood, or the test set is the training set with the points shuffled.

<details><summary>Hint</summary>

`with torch.no_grad():` around `shape_features`. The head is `.double()` if you ever turn the default dtype back to float32. `F.cross_entropy` expects class indices of shape `(N,)`, not one-hot.

</details>


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def shape_features(points):
    """Linearity, planarity, sphericity, principal direction, normal.

    Eigenvalues from the population covariance, largest to smallest:
    L = (l1-l2)/l1, P = (l2-l3)/l1, S = l3/l1.
    torch.linalg.eigh returns eigenvalues ascending.
    """
    points = _t(points)
    direction = torch.tensor([1.0, 0.0, 0.0], dtype=points.dtype)
    normal = torch.tensor([0.0, 0.0, 1.0], dtype=points.dtype)
    if points.shape[0] < 3:
        return 0.0, 0.0, 0.0, direction, normal
    centered = points - points.mean(dim=0)
    cov = (centered.T @ centered) / points.shape[0]
    evals, evecs = torch.linalg.eigh(cov)
    l_min, l_mid, l_max = evals
    if l_max < 1e-15:
        return 0.0, 0.0, 0.0, direction, normal
    linearity = float((l_max - l_mid) / l_max)
    planarity = float((l_mid - l_min) / l_max)
    sphericity = float(l_min / l_max)
    direction = evecs[:, -1].clone()
    normal = evecs[:, 0].clone()
    for vec in (direction, normal):
        k = int(torch.argmax(vec.abs()))
        if vec[k] < 0.0:
            vec *= -1.0
    return linearity, planarity, sphericity, direction, normal

def _shape_cloud(kind, n, g):
    """Synthetic centered cloud. 0 ground, 1 vertical wall, 2 vertical pole."""
    if kind == 0:
        pts = torch.rand(n, 3, generator=g)
        pts[:, 2] = 0.01 * torch.randn(n, generator=g)
    elif kind == 1:
        pts = torch.rand(n, 3, generator=g)
        pts[:, 0] = 0.01 * torch.randn(n, generator=g)
    else:
        pts = torch.zeros(n, 3)
        pts[:, 2] = torch.linspace(0, 1, n)
        pts[:, :2] = 0.01 * torch.randn(n, 2, generator=g)
    return pts - pts.mean(dim=0)

def geometric_feature_row(points):
    """(linearity, planarity, |direction·up|, |normal·up|). No grad through eigh."""
    raise NotImplementedError

def train_feature_head(steps=250, seed=0):
    """Train a linear head on handcrafted neighborhood features.

    Returns the head and held-out accuracy.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def shape_features(points):
    """Linearity, planarity, sphericity, principal direction, normal.

    Eigenvalues from the population covariance, largest to smallest:
    L = (l1-l2)/l1, P = (l2-l3)/l1, S = l3/l1.
    torch.linalg.eigh returns eigenvalues ascending.
    """
    points = _t(points)
    direction = torch.tensor([1.0, 0.0, 0.0], dtype=points.dtype)
    normal = torch.tensor([0.0, 0.0, 1.0], dtype=points.dtype)
    if points.shape[0] < 3:
        return 0.0, 0.0, 0.0, direction, normal
    centered = points - points.mean(dim=0)
    cov = (centered.T @ centered) / points.shape[0]
    evals, evecs = torch.linalg.eigh(cov)
    l_min, l_mid, l_max = evals
    if l_max < 1e-15:
        return 0.0, 0.0, 0.0, direction, normal
    linearity = float((l_max - l_mid) / l_max)
    planarity = float((l_mid - l_min) / l_max)
    sphericity = float(l_min / l_max)
    direction = evecs[:, -1].clone()
    normal = evecs[:, 0].clone()
    for vec in (direction, normal):
        k = int(torch.argmax(vec.abs()))
        if vec[k] < 0.0:
            vec *= -1.0
    return linearity, planarity, sphericity, direction, normal

def geometric_feature_row(points):
    """(linearity, planarity, |direction·up|, |normal·up|). No grad through eigh."""
    with torch.no_grad():
        linearity, planarity, _, direction, normal = shape_features(points)
    return torch.tensor(
        [linearity, planarity, abs(float(direction[2])), abs(float(normal[2]))],
        dtype=DTYPE,
    )

def _shape_cloud(kind, n, g):
    if kind == 0:
        pts = torch.rand(n, 3, generator=g, dtype=DTYPE)
        pts[:, 2] = 0.01 * torch.randn(n, generator=g, dtype=DTYPE)
    elif kind == 1:
        pts = torch.rand(n, 3, generator=g, dtype=DTYPE)
        pts[:, 0] = 0.01 * torch.randn(n, generator=g, dtype=DTYPE)
    else:
        pts = torch.zeros(n, 3, dtype=DTYPE)
        pts[:, 2] = torch.linspace(0, 1, n, dtype=DTYPE)
        pts[:, :2] = 0.01 * torch.randn(n, 2, generator=g, dtype=DTYPE)
    return pts - pts.mean(dim=0)

def train_feature_head(steps=250, seed=0):
    """Train a linear head on handcrafted neighborhood features.

    Classes are ground (0), vertical wall (1), vertical pole (2).
    Returns the head and held-out accuracy.
    """
    g = torch.Generator().manual_seed(seed)
    xs, ys = [], []
    for _ in range(48):
        for kind in range(3):
            xs.append(geometric_feature_row(_shape_cloud(kind, 32, g)))
            ys.append(kind)
    X = torch.stack(xs)
    y = torch.tensor(ys)
    perm = torch.randperm(X.shape[0], generator=g)
    cut = int(0.75 * X.shape[0])
    train_i, test_i = perm[:cut], perm[cut:]
    head = nn.Linear(4, 3).double()
    opt = torch.optim.Adam(head.parameters(), lr=0.08)
    for _ in range(steps):
        opt.zero_grad()
        loss = F.cross_entropy(head(X[train_i]), y[train_i])
        loss.backward()
        opt.step()
    with torch.no_grad():
        acc = (head(X[test_i]).argmax(dim=-1) == y[test_i]).double().mean()
    return head, acc

def check_train_feature_head():
    _, acc = train_feature_head()
    assert float(acc) > 0.9, float(acc)
    print("passed")

check_train_feature_head()


## 3. A tiny PointNet training step

Write `NeighborhoodPointNet` and `train_pointnet(steps=80, seed=0)`.

The module takes `points` of shape `(B, N, 3)`. A shared MLP maps each point from 3 to 64, ReLU, 64 to 64, ReLU. Max-pool over the points. A linear layer maps the pooled 64 to 3 class logits. Max-pool is the permutation-invariant part; say that out loud. A curb neighborhood does not have a point order.

`train_pointnet` builds the module in float64, uses Adam at learning rate `1e-2`, and takes `steps` updates. Each update is a fresh batch from `_shape_cloud`: 8 clouds of each class, 32 points each, labels `(24,)`. Return the module, the first loss, and the last loss. The check requires the last loss to be under half the first.

This is not a model you would ship. It is the shape of the answer if they ask you to train something on a neighborhood: shared weights, a symmetric pool, cross-entropy, a loop you can read.

<details><summary>Hint</summary>

`self.mlp(points)` on `(B, N, 3)` applies the linear layers to the last dimension. `.max(dim=1).values` pools. `F.cross_entropy(logits, labels)` with `labels` a long tensor of class indices.

</details>


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def _shape_cloud(kind, n, g):
    """Synthetic centered cloud. 0 ground, 1 vertical wall, 2 vertical pole."""
    if kind == 0:
        pts = torch.rand(n, 3, generator=g)
        pts[:, 2] = 0.01 * torch.randn(n, generator=g)
    elif kind == 1:
        pts = torch.rand(n, 3, generator=g)
        pts[:, 0] = 0.01 * torch.randn(n, generator=g)
    else:
        pts = torch.zeros(n, 3)
        pts[:, 2] = torch.linspace(0, 1, n)
        pts[:, :2] = 0.01 * torch.randn(n, 2, generator=g)
    return pts - pts.mean(dim=0)

class NeighborhoodPointNet(nn.Module):
    """Shared MLP, max pool, linear head. points are (B, N, 3)."""

    def __init__(self, n_classes=3):
        raise NotImplementedError

    def forward(self, points):
        raise NotImplementedError

def train_pointnet(steps=80, seed=0):
    """Train NeighborhoodPointNet on ground, wall, and pole clouds.

    Returns the module, the first loss, and the last loss.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _shape_cloud(kind, n, g):
    if kind == 0:
        pts = torch.rand(n, 3, generator=g, dtype=DTYPE)
        pts[:, 2] = 0.01 * torch.randn(n, generator=g, dtype=DTYPE)
    elif kind == 1:
        pts = torch.rand(n, 3, generator=g, dtype=DTYPE)
        pts[:, 0] = 0.01 * torch.randn(n, generator=g, dtype=DTYPE)
    else:
        pts = torch.zeros(n, 3, dtype=DTYPE)
        pts[:, 2] = torch.linspace(0, 1, n, dtype=DTYPE)
        pts[:, :2] = 0.01 * torch.randn(n, 2, generator=g, dtype=DTYPE)
    return pts - pts.mean(dim=0)

class NeighborhoodPointNet(nn.Module):
    """Shared MLP, max pool, linear head. points are (B, N, 3)."""

    def __init__(self, n_classes=3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
        )
        self.head = nn.Linear(64, n_classes)

    def forward(self, points):
        pooled = self.mlp(points).max(dim=1).values
        return self.head(pooled)

def train_pointnet(steps=80, seed=0):
    """Train NeighborhoodPointNet on the same three synthetic classes.

    Returns the module, the first loss, and the last loss.
    """
    g = torch.Generator().manual_seed(seed)
    model = NeighborhoodPointNet().double()
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)

    def batch(n_each=8):
        clouds, labels = [], []
        for kind in range(3):
            for _ in range(n_each):
                clouds.append(_shape_cloud(kind, 32, g))
                labels.append(kind)
        return torch.stack(clouds), torch.tensor(labels)

    losses = []
    for step in range(steps):
        pts, labels = batch()
        opt.zero_grad()
        loss = F.cross_entropy(model(pts), labels)
        loss.backward()
        opt.step()
        if step == 0 or step == steps - 1:
            losses.append(float(loss.detach()))
    return model, losses[0], losses[1]

def check_train_pointnet():
    _, first, last = train_pointnet()
    assert last < first * 0.5, (first, last)
    print("passed")

check_train_pointnet()
